In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

In [0]:
customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

In [0]:
orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

In [0]:
orders = orders.withColumn("order_date", to_date(col("order_date")))

In [0]:
customers = customers.filter(col("name").isNotNull() & col("email").isNotNull()) \
    .withColumn("name", trim(col("name")))
orders = orders \
    .filter(col("amount").isNotNull()) \
    .filter(col("amount") > 0)
customers.show()
orders.show()

+-----------+--------+-----------------+---------+
|customer_id|    name|            email|     city|
+-----------+--------+-----------------+---------+
|          1|John Doe| john@example.com|Hyderabad|
|          2|   Alice|alice@example.com|  Chennai|
|          5|     Eva|  eva@example.com|Hyderabad|
|          6|   Frank|frank@example.com|    Delhi|
+-----------+--------+-----------------+---------+

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|     101|          1|2024-01-01|  1000|
|     102|          2|2024-01-02|  2000|
|     104|         99|2024-01-04|  1500|
|     106|          5|2024-01-06|  3000|
|     107|          5|2024-01-07|  3000|
+--------+-----------+----------+------+



In [0]:
inner_join_df = orders.join(customers, "customer_id", "inner")
display(inner_join_df)

customer_id,order_id,order_date,amount,name,email,city
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad


In [0]:
left_join_df = orders.join(customers, "customer_id", "left")
display(left_join_df)

customer_id,order_id,order_date,amount,name,email,city
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai
99,104,2024-01-04,1500,null,null,null
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad


In [0]:
invalid_fk_df = orders.join(customers, "customer_id", "left_anti")
display(invalid_fk_df)

customer_id,order_id,order_date,amount
99,104,2024-01-04,1500


In [0]:
print("Orders Count:", orders.count())
print("Customers Count:", customers.count())
print("Inner Join Count:", inner_join_df.count())
print("Left Join Count:", left_join_df.count())
print("Left Anti Join Count:", invalid_fk_df.count())

Orders Count: 5
Customers Count: 4
Inner Join Count: 4
Left Join Count: 5
Left Anti Join Count: 1


In [0]:
from pyspark.sql.window import Window

In [0]:
df = orders.join(customers, "customer_id", "inner") \
           .filter(col("amount").isNotNull() & (col("amount") > 0))
customer_city_total = df.groupBy("city", "customer_id", "name") \
    .agg(sum("amount").alias("total_spend"))
city_window = Window.orderBy(col("total_spend").desc())

top3_customers = customer_city_total.withColumn(
    "rank", dense_rank().over(city_window)
).filter(col("rank") <= 3)

display(top3_customers)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


city,customer_id,name,total_spend,rank
Hyderabad,5,Eva,6000,1
Chennai,2,Alice,2000,2
Hyderabad,1,John Doe,1000,3


In [0]:
running_window = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, 0)

running_total_df = df.withColumn("running_total", sum("amount").over(running_window))

display(running_total_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,order_id,order_date,amount,name,email,city,running_total
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad,1000
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai,3000
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad,6000
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad,9000


In [0]:
customer_total = df.groupBy("customer_id", "name") \
                   .agg(sum("amount").alias("total_spend"))

rank_window = Window.orderBy(col("total_spend").desc())

ranked_customers = customer_total.withColumn("rank", dense_rank().over(rank_window))

display(ranked_customers)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,name,total_spend,rank
5,Eva,6000,1
2,Alice,2000,2
1,John Doe,1000,3


In [0]:
lag_window = Window.partitionBy("customer_id").orderBy("order_date")

lag_df = df.withColumn("prev_order_amount", lag("amount").over(lag_window))

display(lag_df)

customer_id,order_id,order_date,amount,name,email,city,prev_order_amount
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad,null
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai,null
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad,null
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad,3000


In [0]:
df_with_month = df.withColumn("month", month("order_date"))
display(df_with_month)

customer_id,order_id,order_date,amount,name,email,city,month
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad,1
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai,1
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad,1
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad,1


In [0]:
monthly_sales = df_with_month.groupBy("month") \
    .agg(sum("amount").alias("total_sales")) \
    .orderBy("month")

display(monthly_sales)

month,total_sales
1,9000


In [0]:
lag_window = Window.partitionBy("customer_id").orderBy("order_date")

date_diff_df = df.withColumn(
    "prev_order_date", lag("order_date").over(lag_window)
).withColumn(
    "days_diff", datediff(col("order_date"), col("prev_order_date"))
)

display(date_diff_df)

customer_id,order_id,order_date,amount,name,email,city,prev_order_date,days_diff
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad,null,null
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai,null,null
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad,null,null
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad,2024-01-06,1


In [0]:
trend_df = monthly_sales.withColumn(
    "prev_month_sales",
    lag("total_sales").over(Window.orderBy("month"))
).withColumn(
    "growth",
    col("total_sales") - col("prev_month_sales")
)

display(trend_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


month,total_sales,prev_month_sales,growth
1,9000,null,null


In [0]:
display(df)

customer_id,order_id,order_date,amount,name,email,city
1,101,2024-01-01,1000,John Doe,john@example.com,Hyderabad
2,102,2024-01-02,2000,Alice,alice@example.com,Chennai
5,107,2024-01-07,3000,Eva,eva@example.com,Hyderabad
5,106,2024-01-06,3000,Eva,eva@example.com,Hyderabad


In [0]:
df.write.mode("overwrite").csv("/Volumes/workspace/default/samples_vol/phase6_output.csv")